# NB04 — Visualisasi Lengkap

Hibah Kemdiktisaintek Part 3. SEMUA plot di notebook ini dibangun dari artifact
tersimpan (results/models/, results/predictions/, results/tables/), TIDAK ADA
training ulang di notebook ini. Kalau nb02/nb03 belum selesai jalan, sel-sel di
sini akan gagal (file belum ada), bukan gagal karena bug.

Isi: confusion matrix, ROC curve, PR curve, learning curve, waktu training/
inference, stabilitas antar-fold, t-SNE, critical difference diagram, radar chart,
kurva degradasi vs IR, F1 per kelas, kontribusi meta-feature, diagram kalibrasi,
delta confusion matrix, trade-off efisiensi vs performa.

**Teks di semua figure (title/axis/legend) memakai Bahasa Inggris dan nama model
BERSIH** (arsitektur, bukan nama penulis paper — mis. "IMC-BiLSTM" bukan
"Yang-IMCBiLSTM"), karena figure ini yang berpotensi masuk naskah IEEE Access.
Narasi notebook dan `print()` tetap Bahasa Indonesia.


In [ ]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
warnings_ = __import__("warnings"); warnings_.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import (DATASET_ORDER, DATASETS, TAB_DIR, FIG_DIR, PRED_DIR, MDL_DIR,
                         PLOT_DPI, FIGSIZE_STD, FIGSIZE_WIDE, CLASS_COLORS_LIST, N_FOLDS_FINAL,
                         display_name)
from src.hdmc_multiview import load_dataset, HDMC_TREATMENTS, build_multiview_meta_features_n
from src.metrics import pr_curve_data

sns.set_style("whitegrid")
DS_LABEL = {"sor": "SOR", "otdr": "OTDR", "hardfailure": "HardFailure", "softfailure": "SoftFailure"}

In [ ]:
### Redirect output ke results_clean/ (tidak menimpa results/ asli)
import os
RESULTS_CLEAN_ROOT = os.path.abspath(os.path.join("..", "results_clean"))
for _sub in ["tables", "figures", "models", "predictions"]:
    os.makedirs(os.path.join(RESULTS_CLEAN_ROOT, _sub), exist_ok=True)

_new_tab_dir = os.path.join(RESULTS_CLEAN_ROOT, "tables")
_new_mdl_dir = os.path.join(RESULTS_CLEAN_ROOT, "models")
_new_pred_dir = os.path.join(RESULTS_CLEAN_ROOT, "predictions")
_new_fig_dir = os.path.join(RESULTS_CLEAN_ROOT, "figures")

import src.hdmc_multiview as _m1
import src.comparator_runner as _m2
import src.experiment_runner as _m3
import src.justification_runner as _m4
import src.data_quality as _m5

for _mod in [_m1, _m2, _m3, _m4, _m5]:
    if hasattr(_mod, "TAB_DIR"):
        _mod.TAB_DIR = _new_tab_dir
    if hasattr(_mod, "MDL_DIR"):
        _mod.MDL_DIR = _new_mdl_dir
    if hasattr(_mod, "PRED_DIR"):
        _mod.PRED_DIR = _new_pred_dir

TAB_DIR = _new_tab_dir
MDL_DIR = _new_mdl_dir
PRED_DIR = _new_pred_dir
FIG_DIR = _new_fig_dir

print(f"Output diarahkan ke: {RESULTS_CLEAN_ROOT}")

## Fungsi bantu: baca artifact tersimpan

In [ ]:
def load_all_folds_proba_true(dataset_name, model_or_treatment_dir, n_folds=N_FOLDS_FINAL):
    """Gabungkan y_true/y_proba dari SEMUA fold jadi satu array (evaluasi
    agregat, bukan per fold), dipakai untuk confusion matrix/ROC/PR."""
    all_true, all_proba = [], []
    for i in range(n_folds):
        p = os.path.join(model_or_treatment_dir, f"fold_{i:02d}_val_proba.npy")
        t = os.path.join(model_or_treatment_dir, f"fold_{i:02d}_val_true.npy")
        if not (os.path.exists(p) and os.path.exists(t)):
            continue
        all_proba.append(np.load(p))
        all_true.append(np.load(t))
    if not all_true:
        return None, None
    return np.concatenate(all_true), np.concatenate(all_proba)


def dir_hdmc_meta(dataset_name):
    return os.path.join(PRED_DIR, dataset_name, "meta")


def dir_comparator(dataset_name, model_name):
    return os.path.join(PRED_DIR, dataset_name, model_name)


def dir_backbone_treatment(dataset_name, treatment_label):
    return os.path.join(PRED_DIR, dataset_name, treatment_label)


def reconstruct_full_oof(dataset_name, treatment_label, n_samples, n_classes, n_folds=N_FOLDS_FINAL):
    """Rekonstruksi array OOF PENUH (semua sampel) satu treatment backbone
    dari fold_XX_val_proba.npy + fold_XX_val_idx.npy, dipakai untuk
    membangun ulang meta-feature X_meta HDMC (t-SNE ruang meta-feature)
    tanpa training ulang."""
    d = dir_backbone_treatment(dataset_name, treatment_label)
    oof = np.zeros((n_samples, n_classes), dtype=np.float32)
    for i in range(n_folds):
        p = os.path.join(d, f"fold_{i:02d}_val_proba.npy")
        idx_p = os.path.join(d, f"fold_{i:02d}_val_idx.npy")
        if not (os.path.exists(p) and os.path.exists(idx_p)):
            continue
        proba, idx = np.load(p), np.load(idx_p)
        oof[idx] = proba
    return oof


def plot_confusion_matrix(y_true, y_proba, class_names, title, out_name):
    y_pred = np.argmax(y_proba, axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=FIGSIZE_STD)
    sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues", ax=ax,
               xticklabels=class_names, yticklabels=class_names, cbar_kws={"label": "Row Proportion"})
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
    plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


def plot_roc_curve(y_true, y_proba, class_names, title, out_name):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    if y_bin.shape[1] == 1:
        y_bin = np.hstack([1 - y_bin, y_bin])
    fig, ax = plt.subplots(figsize=FIGSIZE_STD)
    colors = (CLASS_COLORS_LIST * ((len(class_names) // len(CLASS_COLORS_LIST)) + 1))
    for i, name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[i], label=f"{name} (AUC={roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title); ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


def plot_pr_curve(y_true, y_proba, class_names, title, out_name):
    curves = pr_curve_data(y_true, y_proba, class_names)
    fig, ax = plt.subplots(figsize=FIGSIZE_STD)
    colors = (CLASS_COLORS_LIST * ((len(class_names) // len(CLASS_COLORS_LIST)) + 1))
    for i, name in enumerate(class_names):
        ax.plot(curves[name]["recall"], curves[name]["precision"], color=colors[i], label=name)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(title); ax.legend(fontsize=8, loc="lower left")
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")

---
## 1. Confusion matrix + ROC + PR curve, HDMC vs pembanding terkuat, 4 dataset

Agregat SEMUA fold (bukan satu fold), supaya representasi confusion matrix
mencerminkan seluruh data, bukan satu sampel fold saja.

In [ ]:
for ds in DATASET_ORDER:
    class_names = DATASETS[ds]["class_names"]
    y_true, y_proba = load_all_folds_proba_true(ds, dir_hdmc_meta(ds))
    if y_true is None:
        print(f"[{ds}] Prediksi HDMC belum ada, jalankan nb02 dulu.")
        continue
    plot_confusion_matrix(y_true, y_proba, class_names, f"Confusion Matrix — HDMC ({DS_LABEL[ds]})",
                          f"nb04_cm_hdmc_{ds}.png")
    plot_roc_curve(y_true, y_proba, class_names, f"ROC Curve — HDMC ({DS_LABEL[ds]})",
                  f"nb04_roc_hdmc_{ds}.png")
    plot_pr_curve(y_true, y_proba, class_names, f"Precision-Recall Curve — HDMC ({DS_LABEL[ds]})",
                 f"nb04_pr_hdmc_{ds}.png")

### Bandingkan ke pembanding terkuat per dataset (dari nb03)

In [ ]:
comparator_summary_path = os.path.join(TAB_DIR, "comparator_ringkasan_4_dataset.csv")
if os.path.exists(comparator_summary_path):
    df_comp = pd.read_csv(comparator_summary_path)
    for ds in DATASET_ORDER:
        class_names = DATASETS[ds]["class_names"]
        sub = df_comp[df_comp["dataset"] == ds]
        if sub.empty:
            continue
        best_model = sub.loc[sub["macro_f1_mean"].idxmax(), "model"]
        y_true, y_proba = load_all_folds_proba_true(ds, dir_comparator(ds, best_model))
        if y_true is None:
            continue
        plot_confusion_matrix(y_true, y_proba, class_names,
                              f"Confusion Matrix — {display_name(best_model)} (strongest) ({DS_LABEL[ds]})",
                              f"nb04_cm_{best_model}_{ds}.png")
else:
    print("comparator_ringkasan_4_dataset.csv belum ada, jalankan nb03 dulu.")

---
## 2. Learning curve (backbone HDMC, 2 treatment -- SMOTETomek+Focal, None+CE)

Dibaca dari history.json yang tersimpan tiap fold saat training backbone di nb02.

In [ ]:
def plot_learning_curve(dataset_name, treatment_label, fold_idx=0):
    hist_path = os.path.join(MDL_DIR, "backbones", dataset_name, treatment_label,
                             f"fold_{fold_idx:02d}_history.json")
    if not os.path.exists(hist_path):
        print(f"[{dataset_name}/{treatment_label}] history fold {fold_idx} belum ada.")
        return
    with open(hist_path) as f:
        history = json.load(f)
    treatment_clean = treatment_label.replace("_", "+")
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)
    axes[0].plot(history.get("train_loss", []), label="Train Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Training Loss")
    axes[1].plot(history.get("val_f1", []), label="Val Macro-F1", color="darkorange")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro-F1"); axes[1].set_title("Validation Macro-F1")
    fig.suptitle(f"Learning Curve — CNN-BiGRU ({treatment_clean}), {DS_LABEL[dataset_name]}, Fold {fold_idx}")
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"nb04_learning_curve_{dataset_name}_{treatment_label}_fold{fold_idx}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")

for ds in DATASET_ORDER:
    for resampler, loss_type in HDMC_TREATMENTS:
        plot_learning_curve(ds, f"{resampler}_{loss_type}", fold_idx=0)

---
## 3. Waktu training/inference, semua kandidat, 4 dataset

Dibaca dari results/tables/hdmc_ringkasan_4_dataset.csv dan comparator_<dataset>.json
(sudah berisi t_train_s, t_infer_*), tanpa perlu pengukuran ulang.

In [ ]:
rows_time = []
for ds in DATASET_ORDER:
    comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
    if not os.path.exists(comp_path):
        continue
    with open(comp_path) as f:
        comp = json.load(f)
    for model_name, agg in comp.items():
        rows_time.append({
            "dataset": ds, "model": display_name(model_name),
            "t_train_s_mean": (agg.get("timing") or {}).get("t_train_s_mean", np.nan),
        })
df_time = pd.DataFrame(rows_time)
if not df_time.empty:
    for ds in DATASET_ORDER:
        sub = df_time[df_time["dataset"] == ds].sort_values("t_train_s_mean")
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
        ax.barh(sub["model"], sub["t_train_s_mean"], color=CLASS_COLORS_LIST[0])
        ax.set_xlabel("Mean Training Time per Fold (seconds)")
        ax.set_title(f"Training Time, 10 Comparators — {DS_LABEL[ds]}")
        plt.tight_layout()
        out_path = os.path.join(FIG_DIR, f"nb04_waktu_training_{ds}.png")
        plt.savefig(out_path, dpi=PLOT_DPI)
        plt.show()
        print(f"Tersimpan: {out_path}")
else:
    print("Belum ada data timing, jalankan nb03 dulu.")

---
## 4. Stabilitas antar-fold (variance macro-F1)

Box plot macro_f1_per_fold, HDMC vs semua pembanding, tiap dataset.

In [ ]:
for ds in DATASET_ORDER:
    hdmc_json_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
    comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
    if not (os.path.exists(hdmc_json_path) and os.path.exists(comp_path)):
        continue
    with open(hdmc_json_path) as f:
        hdmc_agg = json.load(f)
    with open(comp_path) as f:
        comp = json.load(f)

    data_box, labels_box = [hdmc_agg["macro_f1_per_fold"]], [display_name("HDMC")]
    for model_name, agg in comp.items():
        data_box.append(agg["macro_f1_per_fold"])
        labels_box.append(display_name(model_name))

    fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
    bp = ax.boxplot(data_box, labels=labels_box, patch_artist=True)
    for patch, color in zip(bp["boxes"], [CLASS_COLORS_LIST[i % len(CLASS_COLORS_LIST)] for i in range(len(data_box))]):
        patch.set_facecolor(color); patch.set_alpha(0.6)
    ax.set_ylabel("Macro-F1 per Fold")
    ax.set_title(f"Cross-Fold Stability — {DS_LABEL[ds]}")
    plt.xticks(rotation=30, ha="right")
    
    # Y-Axis Zoom
    min_val = min([min(d) for d in data_box])
    max_val = max([max(d) for d in data_box])
    margin = max((max_val - min_val) * 0.1, 0.01)
    ax.set_ylim(bottom=min_val - margin, top=max_val + margin)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"nb04_stabilitas_{ds}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")

---
## 5. t-SNE ruang meta-feature HDMC (+ beberapa model terpilih)

Direkonstruksi dari OOF proba backbone tersimpan (tanpa retrain).

In [ ]:
from sklearn.manifold import TSNE
from src.config import SEED

def plot_tsne_meta_feature(dataset_name, max_samples=8000, seed=SEED):
    X, y, class_names, groups = load_dataset(dataset_name)
    n_classes = len(class_names)

    oof_list = []
    for resampler, loss_type in HDMC_TREATMENTS:
        label = f"{resampler}_{loss_type}"
        oof = reconstruct_full_oof(dataset_name, label, len(y), n_classes)
        oof_list.append(oof)
    X_meta = build_multiview_meta_features_n(oof_list)

    rng = np.random.default_rng(seed)
    if len(X_meta) > max_samples:
        idx_sub = []
        frac = max_samples / len(X_meta)
        for c in np.unique(y):
            c_idx = np.where(y == c)[0]
            n_take = max(1, int(len(c_idx) * frac))
            idx_sub.append(rng.choice(c_idx, size=min(n_take, len(c_idx)), replace=False))
        idx_sub = np.concatenate(idx_sub)
        X_sub, y_sub = X_meta[idx_sub], y[idx_sub]
    else:
        X_sub, y_sub = X_meta, y

    X_tsne = TSNE(n_components=2, perplexity=min(30, max(5, len(X_sub)//100)),
                  random_state=seed, init="pca").fit_transform(X_sub)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = (CLASS_COLORS_LIST * ((n_classes // len(CLASS_COLORS_LIST)) + 1))
    for i, name in enumerate(class_names):
        mask = y_sub == i
        if mask.sum() == 0:
            continue
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=colors[i], label=name, alpha=0.6, s=12)
    ax.set_xlabel("t-SNE Dimension 1"); ax.set_ylabel("t-SNE Dimension 2")
    ax.set_title(f"t-SNE of HDMC Meta-Feature Space — {DS_LABEL[dataset_name]}")
    ax.legend(fontsize=8, markerscale=2)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"nb04_tsne_meta_{dataset_name}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")

for ds in DATASET_ORDER:
    plot_tsne_meta_feature(ds)

---
## 6. Diagram perbedaan signifikan (Critical Difference, Friedman + Nemenyi)

Standar di paper Q1 yang membandingkan >2 classifier lintas >1 dataset.

CATATAN JUJUR: dengan cuma 4 dataset sebagai unit perbandingan, power statistik
diagram ini terbatas. Tetap informatif untuk visualisasi ringkasan, tapi p-value
per-pasangan dari nb03 (Wilcoxon per fold, per dataset) tetap sumber utama klaim
signifikansi di naskah, bukan diagram ini.

In [ ]:
NEMENYI_Q_ALPHA_005 = {
    2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031,
    9: 3.102, 10: 3.164, 11: 3.219, 12: 3.268, 13: 3.313, 14: 3.354, 15: 3.391,
}

def critical_difference_diagram(rank_means, n_datasets, out_name):
    k = len(rank_means)
    q_alpha = NEMENYI_Q_ALPHA_005.get(k, 3.219)
    cd = q_alpha * np.sqrt(k * (k + 1) / (6 * n_datasets))

    sorted_items = sorted(rank_means.items(), key=lambda kv: kv[1])
    names = [n for n, _ in sorted_items]
    ranks = [r for _, r in sorted_items]

    fig, ax = plt.subplots(figsize=(12, max(4, len(names) * 0.5)))
    ax.scatter(ranks, range(len(names)), s=60, color=CLASS_COLORS_LIST[0], zorder=3)
    for i, (name, r) in enumerate(zip(names, ranks)):
        ax.text(r, i, f"  {name} (rank={r:.2f})", va="center", fontsize=9)

    i = 0
    while i < len(ranks):
        j = i
        while j + 1 < len(ranks) and ranks[j + 1] - ranks[i] <= cd:
            j += 1
        if j > i:
            ax.plot([ranks[i], ranks[j]], [len(names) + 0.5 + i * 0.15] * 2,
                    color="black", linewidth=2)
        i = j + 1

    ax.set_xlabel(f"Average Macro-F1 Rank (CD={cd:.2f}, {n_datasets} datasets, k={k} models)")
    ax.set_yticks([])
    ax.set_title("Critical Difference Diagram — HDMC vs 10 Comparators")
    ax.invert_xaxis()
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.show()
    print(f"Tersimpan: {out_path} (CD={cd:.3f})")
    return cd


hdmc_summary_path = os.path.join(TAB_DIR, "hdmc_ringkasan_4_dataset.csv")
if os.path.exists(comparator_summary_path) and os.path.exists(hdmc_summary_path):
    df_comp = pd.read_csv(comparator_summary_path)
    df_hdmc = pd.read_csv(hdmc_summary_path)
    df_hdmc = df_hdmc.assign(model="HDMC")[["dataset", "model", "macro_f1_mean"]]
    df_all = pd.concat([df_comp[["dataset", "model", "macro_f1_mean"]], df_hdmc], ignore_index=True)
    df_all["model_display"] = df_all["model"].apply(display_name)

    pivot = df_all.pivot(index="dataset", columns="model_display", values="macro_f1_mean")
    ranks = pivot.rank(axis=1, ascending=False)
    rank_means = ranks.mean(axis=0).to_dict()

    critical_difference_diagram(rank_means, n_datasets=len(pivot), out_name="nb04_critical_difference.png")
else:
    print("Data belum lengkap, jalankan nb02+nb03 dulu.")

---
## 7. Radar chart — HDMC vs 3 pembanding terkuat (rata-rata 4 dataset)

In [ ]:
def radar_chart(data_by_model, metric_names, title, out_name):
    n_metrics = len(metric_names)
    angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for i, (model_name, values) in enumerate(data_by_model.items()):
        vals = values + values[:1]
        ax.plot(angles, vals, label=model_name, color=CLASS_COLORS_LIST[i % len(CLASS_COLORS_LIST)])
        ax.fill(angles, vals, alpha=0.08, color=CLASS_COLORS_LIST[i % len(CLASS_COLORS_LIST)])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_names)
    ax.set_ylim(0, 1)
    ax.set_title(title, y=1.08)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.show()
    print(f"Tersimpan: {out_path}")


if os.path.exists(comparator_summary_path) and os.path.exists(hdmc_summary_path):
    rows_full = []
    for ds in DATASET_ORDER:
        hdmc_json_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
        if os.path.exists(hdmc_json_path):
            with open(hdmc_json_path) as f:
                agg = json.load(f)
            rows_full.append({"dataset": ds, "model": "HDMC", "macro_f1_mean": agg["macro_f1_mean"],
                              "bacc_mean": agg["bacc_mean"], "mcc_mean": agg["mcc_mean"],
                              "accuracy_mean": agg["accuracy_mean"]})
        comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
        if os.path.exists(comp_path):
            with open(comp_path) as f:
                comp = json.load(f)
            for name, agg in comp.items():
                rows_full.append({"dataset": ds, "model": name, "macro_f1_mean": agg["macro_f1_mean"],
                                  "bacc_mean": agg["bacc_mean"], "mcc_mean": agg["mcc_mean"],
                                  "accuracy_mean": agg["accuracy_mean"]})
    df_full = pd.DataFrame(rows_full)
    if not df_full.empty:
        df_mean = df_full.groupby("model")[["macro_f1_mean", "bacc_mean", "mcc_mean", "accuracy_mean"]].mean()
        top3 = df_mean.drop(index="HDMC", errors="ignore").sort_values("macro_f1_mean", ascending=False).head(3)
        selected = ["HDMC"] + list(top3.index)

        metric_names = ["Macro-F1", "BAcc", "MCC", "Accuracy"]
        data_by_model = {display_name(m): df_mean.loc[m, ["macro_f1_mean", "bacc_mean", "mcc_mean", "accuracy_mean"]].tolist()
                         for m in selected if m in df_mean.index}
        radar_chart(data_by_model, metric_names,
                   "Mean Across 4 Datasets — HDMC vs Top 3 Comparators", "nb04_radar_chart.png")

---
## 8. Kurva degradasi performa vs Imbalance Ratio

Narasi inti paper (ketahanan terhadap imbalance).

In [ ]:
from src.config import IMBALANCE_RATIO

if os.path.exists(comparator_summary_path) and os.path.exists(hdmc_summary_path):
    df_hdmc = pd.read_csv(hdmc_summary_path).assign(model="HDMC")
    df_comp = pd.read_csv(comparator_summary_path)
    best_per_ds = df_comp.loc[df_comp.groupby("dataset")["macro_f1_mean"].idxmax()]

    fig, ax = plt.subplots(figsize=FIGSIZE_STD)
    ir_vals = [IMBALANCE_RATIO[ds] for ds in DATASET_ORDER]
    hdmc_vals = [df_hdmc[df_hdmc["dataset"] == ds]["macro_f1_mean"].values[0] for ds in DATASET_ORDER]
    best_vals = [best_per_ds[best_per_ds["dataset"] == ds]["macro_f1_mean"].values[0] for ds in DATASET_ORDER]

    ax.plot(ir_vals, hdmc_vals, "o-", label=display_name("HDMC"), color=CLASS_COLORS_LIST[0], linewidth=2, markersize=8)
    ax.plot(ir_vals, best_vals, "s--", label="Best comparator per dataset", color=CLASS_COLORS_LIST[1], linewidth=2, markersize=8)
    for ds, ir, v in zip(DATASET_ORDER, ir_vals, hdmc_vals):
        ax.annotate(DS_LABEL[ds], (ir, v), textcoords="offset points", xytext=(5, 8), fontsize=8)

    ax.set_xscale("log")
    ax.set_xlabel("Imbalance Ratio (log scale)")
    ax.set_ylabel("Macro-F1")
    
    # Y-Axis Zoom
    min_val = min(hdmc_vals + best_vals)
    max_val = max(hdmc_vals + best_vals)
    margin = max((max_val - min_val) * 0.1, 0.01)
    ax.set_ylim(bottom=min_val - margin, top=max_val + margin)
    ax.set_title("Performance Robustness Against Increasing Class Imbalance")
    ax.legend()
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, "nb04_degradasi_vs_ir.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")

---
## 9. Perbandingan F1 per kelas — HDMC vs pembanding terkuat, tiap dataset

In [ ]:
if os.path.exists(comparator_summary_path):
    df_comp = pd.read_csv(comparator_summary_path)
    for ds in DATASET_ORDER:
        class_names = DATASETS[ds]["class_names"]
        hdmc_json_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
        sub = df_comp[df_comp["dataset"] == ds]
        if not os.path.exists(hdmc_json_path) or sub.empty:
            continue
        with open(hdmc_json_path) as f:
            hdmc_agg = json.load(f)
        best_model = sub.loc[sub["macro_f1_mean"].idxmax(), "model"]
        comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
        with open(comp_path) as f:
            comp_agg = json.load(f)[best_model]

        f1_hdmc = [hdmc_agg["per_class"][c]["f1_mean"] for c in class_names]
        f1_comp = [comp_agg["per_class"][c]["f1_mean"] for c in class_names]

        x = np.arange(len(class_names))
        width = 0.35
        fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
        ax.bar(x - width/2, f1_hdmc, width, label=display_name("HDMC"), color=CLASS_COLORS_LIST[0])
        ax.bar(x + width/2, f1_comp, width, label=display_name(best_model), color=CLASS_COLORS_LIST[1])
        ax.set_xticks(x)
        ax.set_xticklabels(class_names, rotation=30, ha="right")
        ax.set_ylabel("F1-score")
        
        # Y-Axis Zoom
        min_val = min(f1_hdmc + f1_comp)
        max_val = max(f1_hdmc + f1_comp)
        margin = max((max_val - min_val) * 0.1, 0.01)
        ax.set_ylim(bottom=max(0, min_val - margin), top=min(1.02, max_val + margin))
        ax.set_title(f"Per-Class F1, HDMC vs {display_name(best_model)} — {DS_LABEL[ds]}")
        ax.legend()
        plt.tight_layout()
        out_path = os.path.join(FIG_DIR, f"nb04_f1_per_kelas_{ds}.png")
        plt.savefig(out_path, dpi=PLOT_DPI)
        plt.show()
        print(f"Tersimpan: {out_path}")

---
## 10. Kontribusi fitur meta-learner (koefisien Logistic Regression)

Interpretabilitas: fitur meta-feature mana yang paling berpengaruh terhadap
keputusan akhir HDMC.

In [ ]:
from src.hdmc_multiview import HDMC_TREATMENTS
import joblib

def nama_kolom_meta_feature(class_names, treatments):
    """Susun nama kolom meta-feature SESUAI urutan build_multiview_meta_features_n."""
    names = []
    for r, l in treatments:
        label = f"{r}+{l}"
        for c in class_names:
            names.append(f"{label}_proba_{c}")
        names.append(f"{label}_margin")
        names.append(f"{label}_entropy")
    for i in range(len(treatments)):
        for j in range(i + 1, len(treatments)):
            a = f"{treatments[i][0]}+{treatments[i][1]}"
            b = f"{treatments[j][0]}+{treatments[j][1]}"
            names.append(f"L1_{a}_vs_{b}")
            names.append(f"KL_{a}_vs_{b}")
    return names


for ds in DATASET_ORDER:
    class_names = DATASETS[ds]["class_names"]
    meta_model_path = os.path.join(MDL_DIR, "meta", ds, "fold_00.joblib")
    if not os.path.exists(meta_model_path):
        continue
    meta_model = joblib.load(meta_model_path)
    if not hasattr(meta_model, "coef_"):
        print(f"[{ds}] meta-learner ({type(meta_model).__name__}) tidak punya .coef_ "
              f"(bukan model linear), lewati plot koefisien.")
        continue

    feat_names = nama_kolom_meta_feature(class_names, HDMC_TREATMENTS)
    coef = meta_model.coef_
    coef_importance = np.abs(coef).mean(axis=0) if coef.ndim > 1 else np.abs(coef)

    top_n = min(15, len(feat_names))
    top_idx = np.argsort(coef_importance)[::-1][:top_n]

    fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
    ax.barh([feat_names[i] for i in top_idx][::-1], coef_importance[top_idx][::-1],
           color=CLASS_COLORS_LIST[0])
    ax.set_xlabel("Mean |Coefficient| (Averaged Across Classes)")
    ax.set_title(f"Top {top_n} Meta-Feature Contributions (Fold 0) — {DS_LABEL[ds]}")
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"nb04_meta_feature_importance_{ds}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")

---
## 11. Diagram kalibrasi (reliability diagram)

Seberapa dekat probabilitas prediksi HDMC dengan frekuensi kejadian sebenarnya.

In [ ]:
from sklearn.calibration import calibration_curve

for ds in DATASET_ORDER:
    class_names = DATASETS[ds]["class_names"]
    y_true, y_proba = load_all_folds_proba_true(ds, dir_hdmc_meta(ds))
    if y_true is None:
        continue

    fig, ax = plt.subplots(figsize=FIGSIZE_STD)
    for i, name in enumerate(class_names):
        y_bin = (y_true == i).astype(int)
        if y_bin.sum() < 5:
            continue
        frac_pos, mean_pred = calibration_curve(y_bin, y_proba[:, i], n_bins=10, strategy="quantile")
        ax.plot(mean_pred, frac_pos, "o-", label=name, color=CLASS_COLORS_LIST[i % len(CLASS_COLORS_LIST)])
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    ax.set_title(f"Calibration Diagram — HDMC ({DS_LABEL[ds]})")
    ax.legend(fontsize=8)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"nb04_kalibrasi_{ds}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")

---
## 12. Confusion matrix delta — HDMC minus pembanding terkuat

In [ ]:
if os.path.exists(comparator_summary_path):
    df_comp = pd.read_csv(comparator_summary_path)
    for ds in DATASET_ORDER:
        class_names = DATASETS[ds]["class_names"]
        sub = df_comp[df_comp["dataset"] == ds]
        if sub.empty:
            continue
        best_model = sub.loc[sub["macro_f1_mean"].idxmax(), "model"]

        y_true_h, y_proba_h = load_all_folds_proba_true(ds, dir_hdmc_meta(ds))
        y_true_c, y_proba_c = load_all_folds_proba_true(ds, dir_comparator(ds, best_model))
        if y_true_h is None or y_true_c is None:
            continue

        cm_h = confusion_matrix(y_true_h, np.argmax(y_proba_h, axis=1), labels=list(range(len(class_names))))
        cm_c = confusion_matrix(y_true_c, np.argmax(y_proba_c, axis=1), labels=list(range(len(class_names))))
        cm_h_norm = cm_h.astype(float) / cm_h.sum(axis=1, keepdims=True).clip(min=1)
        cm_c_norm = cm_c.astype(float) / cm_c.sum(axis=1, keepdims=True).clip(min=1)
        delta = cm_h_norm - cm_c_norm

        fig, ax = plt.subplots(figsize=FIGSIZE_STD)
        sns.heatmap(delta, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
                   xticklabels=class_names, yticklabels=class_names,
                   cbar_kws={"label": "Proportion Difference (HDMC - Comparator)"})
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        ax.set_title(f"Confusion Matrix Delta: HDMC - {display_name(best_model)} ({DS_LABEL[ds]})")
        plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
        plt.tight_layout()
        out_path = os.path.join(FIG_DIR, f"nb04_cm_delta_{ds}.png")
        plt.savefig(out_path, dpi=PLOT_DPI)
        plt.show()
        print(f"Tersimpan: {out_path}")
        print(f"  Diagonal delta (recall per kelas, HDMC - {display_name(best_model)}): "
              f"{dict(zip(class_names, np.round(np.diag(delta), 3)))}")

---
## 13. Efisiensi vs performa (latensi inferensi vs macro-F1)

Memakai waktu inferensi PER FOLD (batch=1).

In [ ]:
rows_eff = []
for ds in DATASET_ORDER:
    hdmc_json_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
    comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
    if os.path.exists(hdmc_json_path):
        with open(hdmc_json_path) as f:
            hdmc_agg = json.load(f)
        oof_t = hdmc_agg.get("oof_timing_per_treatment", {})
        lat_ms = sum(v.get("agg", {}).get("t_infer_bs1_per_sample_ms_mean", 0) for v in oof_t.values())
        rows_eff.append({"dataset": ds, "model": "HDMC", "macro_f1": hdmc_agg["macro_f1_mean"],
                         "latency_ms": lat_ms})
    if os.path.exists(comp_path):
        with open(comp_path) as f:
            comp = json.load(f)
        for name, agg in comp.items():
            t = agg.get("timing") or {}
            lat = t.get("t_infer_bs1_per_sample_ms_mean", t.get("t_infer_total_s_mean", np.nan))
            rows_eff.append({"dataset": ds, "model": name, "macro_f1": agg["macro_f1_mean"], "latency_ms": lat})

df_eff = pd.DataFrame(rows_eff).dropna(subset=["latency_ms"])
if not df_eff.empty:
    for ds in DATASET_ORDER:
        sub = df_eff[df_eff["dataset"] == ds]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=FIGSIZE_STD)
        for _, row in sub.iterrows():
            is_hdmc = row["model"] == "HDMC"
            color = CLASS_COLORS_LIST[0] if is_hdmc else CLASS_COLORS_LIST[2]
            marker = "*" if is_hdmc else "o"
            size = 200 if is_hdmc else 80
            ax.scatter(row["latency_ms"], row["macro_f1"], color=color, marker=marker, s=size, zorder=3)
            ax.annotate(display_name(row["model"]), (row["latency_ms"], row["macro_f1"]),
                       textcoords="offset points", xytext=(5, 5), fontsize=7)
        ax.set_xscale("log")
        ax.set_xlabel("Inference Latency per Sample (ms, log scale, batch=1)")
        ax.set_ylabel("Macro-F1")
        
        # Y-Axis Zoom
        min_val = sub["macro_f1"].min()
        max_val = sub["macro_f1"].max()
        margin = max((max_val - min_val) * 0.1, 0.01)
        ax.set_ylim(bottom=min_val - margin, top=max_val + margin)
        ax.set_title(f"Efficiency-Performance Trade-off — {DS_LABEL[ds]}")
        plt.tight_layout()
        out_path = os.path.join(FIG_DIR, f"nb04_efisiensi_vs_performa_{ds}.png")
        plt.savefig(out_path, dpi=PLOT_DPI)
        plt.show()
        print(f"Tersimpan: {out_path}")
else:
    print("Data timing belum lengkap, jalankan ulang nb02/nb03.")

---
## 14. Macro-F1 Comparison Heatmap (Q1 Standard)
Heatmap F1-score komparatif HDMC vs semua pembanding di 4 dataset.

In [ ]:
rows_f1 = []
for ds in DATASET_ORDER:
    hdmc_json_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
    if os.path.exists(hdmc_json_path):
        with open(hdmc_json_path) as f:
            rows_f1.append({"dataset": DS_LABEL[ds], "model": display_name("HDMC"), "macro_f1": json.load(f)["macro_f1_mean"]})
    comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
    if os.path.exists(comp_path):
        with open(comp_path) as f:
            for name, agg in json.load(f).items():
                rows_f1.append({"dataset": DS_LABEL[ds], "model": display_name(name), "macro_f1": agg["macro_f1_mean"]})

if rows_f1:
    df_f1 = pd.DataFrame(rows_f1)
    pivot_f1 = df_f1.pivot(index="model", columns="dataset", values="macro_f1")
    
    # Sort by average F1 across datasets
    pivot_f1['mean'] = pivot_f1.mean(axis=1)
    pivot_f1 = pivot_f1.sort_values('mean', ascending=False).drop(columns=['mean'])
    
    fig, ax = plt.subplots(figsize=(8, 8))
    sns.heatmap(pivot_f1, annot=True, fmt=".4f", cmap="YlGnBu", ax=ax, cbar_kws={"label": "Macro-F1"}, linewidths=.5)
    ax.set_title("Macro-F1 Comparison Across Datasets", pad=20)
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Model")
    plt.xticks(rotation=15)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, "nb04_macro_f1_heatmap.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


---
## 15. Model Ranking Heatmap (Q1 Standard)
Heatmap ranking performa (1 = terbaik) tiap model di semua dataset.

In [ ]:
if rows_f1:
    pivot_rank = df_f1.pivot(index="model", columns="dataset", values="macro_f1").rank(ascending=False, method='min')
    
    # Sort same as F1
    pivot_rank['mean'] = pivot_rank.mean(axis=1)
    pivot_rank = pivot_rank.sort_values('mean', ascending=True).drop(columns=['mean'])
    
    fig, ax = plt.subplots(figsize=(8, 8))
    sns.heatmap(pivot_rank, annot=True, fmt="g", cmap="coolwarm_r", ax=ax, cbar_kws={"label": "Rank (1 = Best)"}, linewidths=.5)
    ax.set_title("Model Ranking Across Datasets", pad=20)
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Model")
    plt.xticks(rotation=15)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, "nb04_rank_heatmap.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


---
## 16. Violin Plot Stabilitas (Q1 Standard)
Menyatukan stabilitas (distribusi F1 per fold) seluruh dataset dalam satu grafik.

In [ ]:
import matplotlib.patches as mpatches
rows_violin = []
for ds in DATASET_ORDER:
    hdmc_json_path = os.path.join(TAB_DIR, f"{ds}_hdmc_final_metrics.json")
    comp_path = os.path.join(TAB_DIR, f"comparator_{ds}.json")
    if not (os.path.exists(hdmc_json_path) and os.path.exists(comp_path)):
        continue
    with open(hdmc_json_path) as f:
        for val in json.load(f)["macro_f1_per_fold"]:
            rows_violin.append({"dataset": DS_LABEL[ds], "model": display_name("HDMC"), "f1": val, "cat": "proposed"})
    with open(comp_path) as f:
        for name, agg in json.load(f).items():
            cat = "external" if "Baseline" not in name else "baseline"
            for val in agg["macro_f1_per_fold"]:
                rows_violin.append({"dataset": DS_LABEL[ds], "model": display_name(name), "f1": val, "cat": cat})

if rows_violin:
    df_violin = pd.DataFrame(rows_violin)
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.violinplot(data=df_violin, x="dataset", y="f1", hue="model", inner="point", linewidth=1, ax=ax)
    ax.set_title("Distribution of Macro-F1 across 10-Folds (Cross-Dataset)")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Macro-F1")
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, "nb04_violin_stabilitas_all.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


## 17. Architecture & Workflow Diagrams (B&W Q1 Format)
Digenerate khusus agar serupa dengan format alur hitam-putih (bersih, rapi) menggunakan matplotlib shapes.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches

OUT_DIR = "../results/figures"
os.makedirs(OUT_DIR, exist_ok=True)
DPI = 300

def draw_box(ax, center_x, center_y, width, height, text, fontsize=10, bold_title=False):
    box = patches.Rectangle((center_x - width/2, center_y - height/2), width, height,
                            edgecolor='black', facecolor='white', lw=1.5, zorder=2)
    ax.add_patch(box)
    lines = text.split('\n')
    if bold_title and len(lines) > 1:
        ax.text(center_x, center_y + height/4, lines[0], ha='center', va='center', fontsize=fontsize, fontweight='bold', zorder=3)
        ax.text(center_x, center_y - height/8, '\n'.join(lines[1:]), ha='center', va='center', fontsize=fontsize-1, zorder=3)
    else:
        ax.text(center_x, center_y, text, ha='center', va='center', fontsize=fontsize, fontweight='bold' if bold_title else 'normal', zorder=3)
    return center_x, center_y - height/2, center_y + height/2, center_x - width/2, center_x + width/2

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", lw=1.5, color="black"), zorder=1)

# 1. OVERALL FRAMEWORK
def plot_overall_framework():
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')
    bx1, by1_b, _, _, _ = draw_box(ax, 50, 90, 45, 10, "4 Datasets\nOTDR, Hardfailure, Softfailure, SOR", bold_title=True)
    bx2, by2_b, bx2_t, _, _ = draw_box(ax, 50, 72, 45, 10, "Imbalanced-Aware\nMeta-Classification Framework", bold_title=True)
    draw_arrow(ax, bx1, by1_b, bx2, bx2_t)
    w, h = 30, 12
    bx3, by3_b, bx3_t, _, _ = draw_box(ax, 17, 50, w, h, "Data Processing Level\nSMOTETomek, None", bold_title=True)
    bx4, by4_b, bx4_t, _, _ = draw_box(ax, 50, 50, w, h, "Model Construction Level\nCNN-BiGRU + Multi-view", bold_title=True)
    bx5, by5_b, bx5_t, _, _ = draw_box(ax, 83, 50, w, h, "Training Optimization Level\nFocal, CE", bold_title=True)
    mid_y_split = (by2_b + bx4_t) / 2
    ax.plot([50, 50], [by2_b, mid_y_split], color="black", lw=1.5, zorder=1); ax.plot([17, 83], [mid_y_split, mid_y_split], color="black", lw=1.5, zorder=1)
    draw_arrow(ax, 17, mid_y_split, bx3, bx3_t); draw_arrow(ax, 50, mid_y_split, bx4, bx4_t); draw_arrow(ax, 83, mid_y_split, bx5, bx5_t)
    bx6, by6_b, bx6_t, _, _ = draw_box(ax, 50, 30, 45, 8, "Class Prediction\nPer dataset", bold_title=True)
    mid_y_merge = (by4_b + bx6_t) / 2
    ax.plot([17, 17], [by3_b, mid_y_merge], color="black", lw=1.5, zorder=1); ax.plot([50, 50], [by4_b, mid_y_merge], color="black", lw=1.5, zorder=1); ax.plot([83, 83], [by5_b, mid_y_merge], color="black", lw=1.5, zorder=1)
    ax.plot([17, 83], [mid_y_merge, mid_y_merge], color="black", lw=1.5, zorder=1)
    draw_arrow(ax, 50, mid_y_merge, bx6, bx6_t)
    bx7, _, bx7_t, _, _ = draw_box(ax, 50, 15, 45, 10, "Evaluation\nMacro-F1, BAcc, MCC", bold_title=True)
    draw_arrow(ax, bx6, by6_b, bx7, bx7_t)
    plt.savefig(os.path.join(OUT_DIR, "nb04_01_overall_framework_bw.png"), dpi=DPI, bbox_inches='tight')
    plt.show()

# 2. HDMC WORKFLOW
def plot_hdmc_workflow():
    fig, ax = plt.subplots(figsize=(10, 12))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')
    w, h = 32, 10
    t1x, t1b, _, _, _ = draw_box(ax, 30, 90, w, h, "Treatment A\nSMOTETomek + Focal", bold_title=True)
    t2x, t2b, _, _, _ = draw_box(ax, 70, 90, w, h, "Treatment B\nNone + CE", bold_title=True)
    b1x, b1b, b1t, _, _ = draw_box(ax, 30, 73, w, h, "CNN+BiGRU", bold_title=True)
    b2x, b2b, b2t, _, _ = draw_box(ax, 70, 73, w, h, "CNN+BiGRU", bold_title=True)
    draw_arrow(ax, t1x, t1b, b1x, b1t); draw_arrow(ax, t2x, t2b, b2x, b2t)
    o1x, o1b, o1t, _, _ = draw_box(ax, 30, 56, w, h, "OOF probability", bold_title=True)
    o2x, o2b, o2t, _, _ = draw_box(ax, 70, 56, w, h, "OOF probability", bold_title=True)
    draw_arrow(ax, b1x, b1b, o1x, o1t); draw_arrow(ax, b2x, b2b, o2x, o2t)
    mx, mb, mt, _, _ = draw_box(ax, 50, 37, 45, 10, "Multi-view feature\nConcatenation + disagreement score", bold_title=True)
    mid_y_merge = (o1b + mt) / 2
    ax.plot([30, 30], [o1b, mid_y_merge], color="black", lw=1.5, zorder=1); ax.plot([70, 70], [o2b, mid_y_merge], color="black", lw=1.5, zorder=1)
    ax.plot([30, 70], [mid_y_merge, mid_y_merge], color="black", lw=1.5, zorder=1)
    draw_arrow(ax, 50, mid_y_merge, mx, mt)
    lx, lb, lt, _, _ = draw_box(ax, 50, 20, 45, 10, "Meta-learner\nLogistic Regression", bold_title=True)
    draw_arrow(ax, mx, mb, lx, lt)
    px, _, pt, _, _ = draw_box(ax, 50, 5, 45, 8, "Class prediction", bold_title=True)
    draw_arrow(ax, lx, lb, px, pt)
    plt.savefig(os.path.join(OUT_DIR, "nb04_02_hdmc_workflow_bw.png"), dpi=DPI, bbox_inches='tight')
    plt.show()

# 3. CNN-BiGRU NEURON ARCHITECTURE (tidak berubah, generik)
def plot_neural_network():
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlim(-5, 105); ax.set_ylim(-10, 110); ax.axis('off')
    layers = [
        {"name": "Input\n(1D Seq)", "n": 8, "x": 10},
        {"name": "Conv1D+ReLU\n(Filters=64)", "n": 8, "x": 30},
        {"name": "BiGRU\n(Forw+Backw)", "n": 10, "x": 50},
        {"name": "GlobalPool\n+ Dropout", "n": 6, "x": 70},
        {"name": "FC Output\n(Softmax)", "n": 4, "x": 90},
    ]
    node_positions = []
    for layer in layers:
        n, x = layer["n"], layer["x"]
        spacing = 100 / (n + 1)
        nodes = [(x, i * spacing) for i in range(1, n + 1)]
        node_positions.append(nodes)
        ax.text(x, -5, layer["name"], ha='center', va='top', fontsize=9, fontweight='bold')
    for i in range(len(node_positions) - 1):
        for sn in node_positions[i]:
            for en in node_positions[i+1]:
                ax.plot([sn[0], en[0]], [sn[1], en[1]], color='lightgray', lw=0.4, alpha=0.8, zorder=1)
    for nodes in node_positions:
        for nx, ny in nodes:
            circle = patches.Circle((nx, ny), radius=2, edgecolor='black', facecolor='white', lw=1.2, zorder=2)
            ax.add_patch(circle)
    plt.title("Backbone Architecture: CNN-BiGRU", pad=20, fontsize=12, fontweight='bold')
    plt.savefig(os.path.join(OUT_DIR, "nb04_03_network_neuron_bw.png"), dpi=DPI, bbox_inches='tight')
    plt.show()

# 4. BIGRU MECHANISM DETAIL (tidak berubah, generik)
def plot_bigru_mechanism():
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')
    ix, ib, it, _, _ = draw_box(ax, 50, 85, 40, 10, "Extracted Features from Conv1D", bold_title=True)
    f_x, f_b, f_t, _, _ = draw_box(ax, 30, 55, 30, 15, "Forward GRU\n(Sequence T \u2192)", bold_title=True)
    b_x, b_b, b_t, _, _ = draw_box(ax, 70, 55, 30, 15, "Backward GRU\n(\u2190 Sequence T)", bold_title=True)
    mid_y_in = (ib + f_t) / 2
    ax.plot([50, 50], [ib, mid_y_in], color="black", lw=1.5, zorder=1); ax.plot([30, 70], [mid_y_in, mid_y_in], color="black", lw=1.5, zorder=1)
    draw_arrow(ax, 30, mid_y_in, f_x, f_t); draw_arrow(ax, 70, mid_y_in, b_x, b_t)
    cx, cb, ct, _, _ = draw_box(ax, 50, 25, 35, 12, "Concatenation Layer\n[Forward ; Backward]", bold_title=True)
    mid_y_out = (f_b + ct) / 2
    ax.plot([30, 30], [f_b, mid_y_out], color="black", lw=1.5, zorder=1); ax.plot([70, 70], [b_b, mid_y_out], color="black", lw=1.5, zorder=1)
    ax.plot([30, 70], [mid_y_out, mid_y_out], color="black", lw=1.5, zorder=1); draw_arrow(ax, 50, mid_y_out, cx, ct)
    plt.savefig(os.path.join(OUT_DIR, "nb04_04_bigru_detail_bw.png"), dpi=DPI, bbox_inches='tight')
    plt.show()

# 5. MULTI-VIEW CONCATENATION DETAIL
def plot_multiview_concatenation():
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')
    draw_box(ax, 30, 80, 28, 12, "OOF Treatment A\n(N x C)", bold_title=True)
    draw_box(ax, 70, 80, 28, 12, "OOF Treatment B\n(N x C)", bold_title=True)
    dx, db, dt, _, _ = draw_box(ax, 25, 45, 26, 16, "Disagreement\nScore (L1/KL)\n(N x 1)", bold_title=True)
    cx, cb, ct, _, _ = draw_box(ax, 63, 45, 44, 16, "Feature Concatenation\n(N x 2C)", bold_title=True)
    ax.plot([30, 30], [74, 60], color="black", lw=1.5, zorder=1); ax.plot([70, 70], [74, 60], color="black", lw=1.5, zorder=1)
    ax.plot([30, 70], [60, 60], color="black", lw=1.5, zorder=1)
    draw_arrow(ax, 63, 60, cx, ct); draw_arrow(ax, 25, 60, dx, dt)
    fx, _, ft, _, _ = draw_box(ax, 44, 15, 50, 12, "Final Multi-View Meta-Feature\n(N x (2C + 1)) \u2192 To LogReg", bold_title=True)
    ax.plot([25, 25], [db, 28], color="black", lw=1.5, zorder=1); ax.plot([63, 63], [cb, 28], color="black", lw=1.5, zorder=1)
    ax.plot([25, 63], [28, 28], color="black", lw=1.5, zorder=1); draw_arrow(ax, 44, 28, fx, ft)
    plt.savefig(os.path.join(OUT_DIR, "nb04_05_multiview_detail_bw.png"), dpi=DPI, bbox_inches='tight')
    plt.show()

# 6. DATA PROCESSING FLOWCHART
def plot_data_resampling():
    fig, ax = plt.subplots(figsize=(8, 10))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')
    ix, ib, it, _, _ = draw_box(ax, 50, 90, 40, 12, "Imbalanced Dataset\nMajority >> Minority", bold_title=True)
    sx, sb, st, _, _ = draw_box(ax, 50, 65, 45, 15, "Oversampling (SMOTE)\nSynthesize new minority instances\nbased on k-nearest neighbors", bold_title=True)
    draw_arrow(ax, ix, ib, sx, st)
    tx, tb, tt, _, _ = draw_box(ax, 50, 35, 45, 15, "Undersampling (Tomek Links)\nRemove overlapping majority\ninstances near the boundary", bold_title=True)
    draw_arrow(ax, sx, sb, tx, tt)
    fx, fb, ft, _, _ = draw_box(ax, 50, 10, 40, 12, "Balanced Dataset\nReady for Model Construction", bold_title=True)
    draw_arrow(ax, tx, tb, fx, ft)
    plt.savefig(os.path.join(OUT_DIR, "nb04_06_data_processing_bw.png"), dpi=DPI, bbox_inches='tight')
    plt.show()

# 7. CNN-BiGRU BACKBONE (tidak berubah, generik)
def plot_cnn_bigru_backbone():
    fig, ax = plt.subplots(figsize=(10, 9))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')
    plt.title("Backbone Architecture: CNN-BiGRU (One Treatment Branch)",
              pad=15, fontsize=13, fontweight='bold')
    layers = [
        "Input Sequence\n(1D Optical Signal)",
        "Conv1D + BatchNorm + ReLU\n(Feature Extraction)",
        "BiGRU Layer\n(Forward & Backward Sequence Modeling)",
        "Global Max Pooling / Flatten",
        "Dropout (Regularization)",
        "Fully Connected Output Layer\n(n_classes, Softmax)",
    ]
    w, h = 65, 9
    y_top = 90
    gap = 15
    prev_bawah = None
    for i, text in enumerate(layers):
        cy = y_top - i * gap
        cx, bawah, atas, _, _ = draw_box(ax, 50, cy, w, h, text, fontsize=10, bold_title=True)
        if prev_bawah is not None:
            draw_arrow(ax, 50, prev_bawah, 50, atas)
        prev_bawah = bawah
    plt.savefig(os.path.join(OUT_DIR, "nb04_cnn_bigru_backbone.png"), dpi=DPI, bbox_inches='tight')
    plt.show()

# EXECUTE ALL
plot_overall_framework()
plot_hdmc_workflow()
plot_neural_network()
plot_bigru_mechanism()
plot_multiview_concatenation()
plot_data_resampling()
plot_cnn_bigru_backbone()
print("SELESAI! 7 gambar berhasil ditampilkan (2-treatment: SMOTETomek+Focal, None+CE).")